# Notebook 08 — Baseline Model

Establish a performance floor using only cyclic time features and region encoding.
The full model (notebook 09) must beat this by at least 10% MAE on the 2022 validation set.

**Reads:** `final_dataset.parquet`  
**Writes:** `models/baseline_model.pkl`, `data/processed/baseline_metrics.json`

In [1]:
import pandas as pd
import numpy as np
import json
import pickle
from pathlib import Path
from sklearn.linear_model import PoissonRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

weekly = pd.read_parquet(PROCESSED_DIR / "final_dataset.parquet")
print(f"Dataset: {len(weekly)} rows")

Dataset: 5016 rows


## Train / Validation / Test Split

**NEVER shuffle time series data.** Split chronologically:
- Train: 2016–2021
- Validation: 2022
- Test: 2023–2024 (held out — do not evaluate until model is finalised)

Note: 2015 is excluded from training due to the 52-week lag warmup.

In [2]:
train = weekly[weekly["week_start"] < "2022-01-01"]
val = weekly[(weekly["week_start"] >= "2022-01-01") & (weekly["week_start"] < "2023-01-01")]
test = weekly[weekly["week_start"] >= "2023-01-01"]  # HELD OUT -- do not use yet

print(f"Train: {len(train)} rows ({train['week_start'].min()} to {train['week_start'].max()})")
print(f"Val:   {len(val)} rows ({val['week_start'].min()} to {val['week_start'].max()})")
print(f"Test:  {len(test)} rows ({test['week_start'].min()} to {test['week_start'].max()})")

Train: 3132 rows (2017-01-02 00:00:00 to 2021-12-27 00:00:00)
Val:   624 rows (2022-01-03 00:00:00 to 2022-12-26 00:00:00)
Test:  1260 rows (2023-01-02 00:00:00 to 2024-12-30 00:00:00)


## Baseline: Poisson GLM with Time Features Only

In [3]:
REGION_COLS = [c for c in weekly.columns if c.startswith("region_")]

# Per-day weather columns (days 0–6, max and min, plus deltas)
WEATHER_COLS = [
    col
    for d in range(7)
    for col in [
        f"temperature_2m_max_{d}_days_prior_mean",
        f"temperature_2m_min_{d}_days_prior_mean",
        f"temperature_2m_max_{d}_days_prior_delta_mean",
        f"temperature_2m_min_{d}_days_prior_delta_mean",
    ]
    if col in weekly.columns
]

TIME_FEATURES = [
    "month_sin", "month_cos", "dayofyear_sin", "dayofyear_cos",
    "season_sin", "season_cos",
    "moon_age",
    "plankton_density",
    "temp_7day_mean_max_mean",
    "temp_7day_max_max_mean",
    "temp_7day_min_min_mean",
    "temp_7day_range_mean",
    *WEATHER_COLS,
    *REGION_COLS,
]
TIME_FEATURES = [c for c in TIME_FEATURES if c in weekly.columns]
print(f"Using {len(TIME_FEATURES)} features: {TIME_FEATURES}")

baseline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", PoissonRegressor(alpha=1.0, max_iter=300)),
])
baseline.fit(train[TIME_FEATURES], train["stranding_count"])

train_preds = baseline.predict(train[TIME_FEATURES])
val_preds = baseline.predict(val[TIME_FEATURES])

Using 50 features: ['month_sin', 'month_cos', 'dayofyear_sin', 'dayofyear_cos', 'season_sin', 'season_cos', 'moon_age', 'plankton_density', 'temp_7day_mean_max_mean', 'temp_7day_max_max_mean', 'temp_7day_min_min_mean', 'temp_7day_range_mean', 'temperature_2m_max_0_days_prior_mean', 'temperature_2m_min_0_days_prior_mean', 'temperature_2m_max_1_days_prior_mean', 'temperature_2m_min_1_days_prior_mean', 'temperature_2m_max_1_days_prior_delta_mean', 'temperature_2m_min_1_days_prior_delta_mean', 'temperature_2m_max_2_days_prior_mean', 'temperature_2m_min_2_days_prior_mean', 'temperature_2m_max_2_days_prior_delta_mean', 'temperature_2m_min_2_days_prior_delta_mean', 'temperature_2m_max_3_days_prior_mean', 'temperature_2m_min_3_days_prior_mean', 'temperature_2m_max_3_days_prior_delta_mean', 'temperature_2m_min_3_days_prior_delta_mean', 'temperature_2m_max_4_days_prior_mean', 'temperature_2m_min_4_days_prior_mean', 'temperature_2m_max_4_days_prior_delta_mean', 'temperature_2m_min_4_days_prior_de

/Users/daivik/Development/se-coast-strandings/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['temperature_2m_max_6_days_prior_delta_mean'
 'temperature_2m_min_6_days_prior_delta_mean']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/daivik/Development/se-coast-strandings/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['temperature_2m_max_6_days_prior_delta_mean'
 'temperature_2m_min_6_days_prior_delta_mean']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/daivik/Development/se-coast-strandings/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['temperature_2m_max_6_days_prior_delta_mean'
 'temperature_2m_min_6_days_prior_delta_mean']. At least one n

## Evaluate Baseline

In [4]:
train_mae = mean_absolute_error(train["stranding_count"], train_preds)
val_mae = mean_absolute_error(val["stranding_count"], val_preds)
val_rmse = np.sqrt(mean_squared_error(val["stranding_count"], val_preds))

# Mean Poisson Deviance
def poisson_deviance(y_true, y_pred):
    y_pred = np.maximum(y_pred, 1e-10)  # avoid log(0)
    return 2 * np.mean(y_true * np.log(np.maximum(y_true, 1e-10) / y_pred) - (y_true - y_pred))

val_deviance = poisson_deviance(val["stranding_count"].values, val_preds)

print("=== Baseline Model (Poisson GLM) ===")
print(f"Train MAE:           {train_mae:.4f}")
print(f"Val MAE:             {val_mae:.4f}")
print(f"Val RMSE:            {val_rmse:.4f}")
print(f"Val Mean Poisson Dev: {val_deviance:.4f}")

=== Baseline Model (Poisson GLM) ===
Train MAE:           0.5792
Val MAE:             0.5594
Val RMSE:            0.7539
Val Mean Poisson Dev: 1.0726


## Save Model and Metrics

In [5]:
# Save model
with open(MODELS_DIR / "baseline_model.pkl", "wb") as f:
    pickle.dump(baseline, f)

# Save metrics
metrics = {
    "model": "PoissonRegressor (baseline)",
    "features": TIME_FEATURES,
    "train_mae": float(train_mae),
    "val_mae": float(val_mae),
    "val_rmse": float(val_rmse),
    "val_mean_poisson_deviance": float(val_deviance),
    "train_rows": len(train),
    "val_rows": len(val),
}

with open(PROCESSED_DIR / "baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved baseline_model.pkl and baseline_metrics.json")
print(f"\nBaseline Val MAE: {val_mae:.4f} -- full model must achieve < {val_mae * 0.9:.4f}")

Saved baseline_model.pkl and baseline_metrics.json

Baseline Val MAE: 0.5594 -- full model must achieve < 0.5035
